In [ ]:
import json
import cv2
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches


# ============================================================
# MAPPING TYPES
# ============================================================
TYPE_MAPPING = {
    'button'      : 'button',
    'textbox'     : 'text',
    'dropdown'    : 'dropdown',
    'checkbox'    : 'checkbox',
    'radio button': 'checkbox',
    'popup'       : 'button',
}

# ============================================================
# CONVERSION PIXELS → POURCENTAGES
# ============================================================
def bbox_to_percent(bbox, img_width, img_height):
    x1, y1, x2, y2 = bbox
    return {
        "x"            : round((x1 / img_width)        * 100, 2),
        "y"            : round((y1 / img_height)       * 100, 2),
        "width"        : round(((x2 - x1) / img_width) * 100, 2),
        "height"       : round(((y2 - y1) / img_height)* 100, 2),
        "canvas_width" : img_width,
        "canvas_height": img_height
    }

# ============================================================
# TROUVER L'ÉLÉMENT CLIQUÉ
# ============================================================
def find_clicked_element(elements, click_x, click_y):
    closest  = None
    min_dist = float('inf')

    for el in elements:
        cx, cy = el['center']
        dist   = ((cx - click_x)**2 + (cy - click_y)**2) ** 0.5
        if dist < min_dist:
            min_dist = dist
            closest  = el

    return closest




In [ ]:
def generate_steps(events, screenshots_dir):
    
    interactive_images = []
    prev_elements      = None
    prev_screen        = None

    for i, event in enumerate(events):
        
        current_screen = os.path.join(screenshots_dir, event['screenshot'])
        
        # Analyser le screen actuel
        elements, tb_raw, tb_merged = analyze_screenshot_smart(
            current_screen, prev_screen, prev_elements
        )

        # ============================================
        # STEP 1 — L'élément cliqué sur ce screen
        # ============================================
        clicked = find_clicked_element(elements, event['x'], event['y'])
        
        if clicked:
            img    = cv2.imread(current_screen)
            img_h, img_w = img.shape[:2]
            position = bbox_to_percent(clicked['bbox'], img_w, img_h)

            interactive_images.append({
                "image_id" : f"step_{i}_click",
                "sequence" : len(interactive_images),
                "image"    : event['screenshot'],
                "instruction": f"Cliquez sur {clicked['label'] or clicked['value']}",
                "screenshot_width" : img_w,
                "screenshot_height": img_h,
                "fields"   : [{
                    "field_id"  : f"step_{i}_field_click",
                    "name"      : clicked['label'] or clicked['value'],
                    "field_type": TYPE_MAPPING.get(clicked['type'], 'button'),
                    "active"    : True,
                    "required"  : True,
                    "colored"   : True,
                    "order"     : 0,
                    "position"  : position,
                    "field_data": {
                        "description"    : clicked['label'] or "",
                        "instruction"    : f"Cliquez sur {clicked['label'] or clicked['value']}",
                        "check_response" : False,
                        "correct_answers": [],
                    }
                }]
            })

        # ============================================
        # STEP 2 — Valeur saisie (si screen suivant existe)
        # ============================================
        if clicked and clicked['type'] == 'textbox' and i + 1 < len(events):
            
            next_screen_path = os.path.join(
                screenshots_dir, events[i + 1]['screenshot']
            )
            
            # Analyser le screen SUIVANT pour lire la valeur saisie
            next_elements, _, _ = analyze_screenshot_smart(
                next_screen_path, current_screen, elements
            )

            # Trouver le même textbox sur le screen suivant
            same_textbox = find_same_element(
                clicked, next_elements
            )

            if same_textbox and same_textbox['value']:
                
                img    = cv2.imread(next_screen_path)
                img_h, img_w = img.shape[:2]
                position = bbox_to_percent(same_textbox['bbox'], img_w, img_h)

                interactive_images.append({
                    "image_id" : f"step_{i}_type",
                    "sequence" : len(interactive_images),
                    "image"    : events[i + 1]['screenshot'],
                    "instruction": f"Tapez {same_textbox['value']} dans {clicked['label']}",
                    "screenshot_width" : img_w,
                    "screenshot_height": img_h,
                    "fields"   : [{
                        "field_id"  : f"step_{i}_field_type",
                        "name"      : clicked['label'],
                        "field_type": "text",
                        "active"    : True,
                        "required"  : True,
                        "colored"   : True,
                        "order"     : 0,
                        "position"  : position,
                        "field_data": {
                            "description"    : clicked['label'],
                            "instruction"    : f"Tapez {same_textbox['value']}",
                            "check_response" : True,
                            "correct_answers": [same_textbox['value']],
                        }
                    }]
                })

        prev_screen   = current_screen
        prev_elements = elements

    return interactive_images


def find_same_element(original_el, next_elements):
    """
    Retrouve le même textbox sur le screen suivant.
    Cherche par proximité de position et même label.
    """
    ox1, oy1, ox2, oy2 = original_el['bbox']
    o_center_x = (ox1 + ox2) / 2
    o_center_y = (oy1 + oy2) / 2

    best    = None
    min_dist = float('inf')

    for el in next_elements:
        if el['type'] != 'textbox':
            continue

        # Même label
        if el['label'] != original_el['label']:
            continue

        cx, cy = el['center']
        dist   = ((cx - o_center_x)**2 + (cy - o_center_y)**2) ** 0.5

        if dist < min_dist and dist < 50:
            min_dist = dist
            best     = el

    return best



def generate_ederest_json_final(events_path, screenshots_dir, output_path):
    
    with open(events_path, 'r', encoding='utf-8') as f:
        events = json.load(f)

    # Générer tous les steps
    interactive_images = generate_steps(events, screenshots_dir)

    # Wrapper Ederest
    result = {
        "title"            : "Simulation SAP",
        "platform"         : "SAP",
        "language"         : "fr",
        "formation_type"   : "TUTORIAL",
        "difficulty"       : "beginner",
        "total_step_count" : len(interactive_images),
        "total_image_count": len(interactive_images),
        "collections"      : [{
            "collection_id"     : "collection_1",
            "collection_type"   : "interactive",
            "sequence"          : 0,
            "required"          : True,
            "interactive_images": interactive_images
        }]
    }

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    print(f" JSON généré : {output_path}")
    print(f"   {len(interactive_images)} steps générés")
    return result



In [ ]:
# Lancer
result = generate_ederest_json_final(
    events_path     = '/content/drive/MyDrive/PAbatchesfolder/test_pipeline_3/events3.json',
    screenshots_dir = '/content/drive/MyDrive/PAbatchesfolder/test_pipeline_3/',
    output_path     = '/content/drive/MyDrive/PAbatchesfolder/test_pipeline_3/simulation_ederest.json'
)